In [1]:
import torch
import transformers

from transformers import pipeline

In [3]:
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1

summarizer = pipeline(
    task="summarization",
    model="facebook/bart-large-cnn",
    framework="pt",
    device=device
)

print("Summarization model loaded successfully!")

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

d:\RetailMind\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SUBHOJIT\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


Summarization model loaded successfully!


In [5]:
reviews = """
The battery life of this smartphone is excellent.
I can easily use it for two days without charging.
The display quality is amazing and colors look vibrant.
The camera takes good pictures during daylight.
However, the phone heats up during heavy gaming.
Gaming performance is also slightly disappointing.
The phone has a premium design and feels comfortable.
Overall, I am satisfied with the purchase."""

In [6]:
summary = summarizer(
    reviews,
    max_length=80,
    min_length=25,
    do_sample=False
)

print("Generated Summary:")
print(summary[0]["summary_text"])

Generated Summary:
The phone has a premium design and feels comfortable. The display quality is amazing and colors look vibrant. However, the phone heats up during heavy gaming.


In [7]:
customer_reviews = [
    "Battery life is excellent and lasts almost two days.",
    "The display is bright and has beautiful colors.",
    "Camera quality is impressive in daylight.",
    "The phone heats up during heavy gaming.",
    "Gaming performance could have been better.",
    "The phone has a premium design.",
    "Charging speed is quite fast.",
    "Overall, a good phone for everyday usage."
]

In [8]:
combined_reviews = " ".join(customer_reviews)

print(combined_reviews)

Battery life is excellent and lasts almost two days. The display is bright and has beautiful colors. Camera quality is impressive in daylight. The phone heats up during heavy gaming. Gaming performance could have been better. The phone has a premium design. Charging speed is quite fast. Overall, a good phone for everyday usage.


In [9]:
summary = summarizer(
    combined_reviews,
    max_length=80,
    min_length=25,
    do_sample=False,
    truncation=True
)

print(summary[0]["summary_text"])

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Your max_length is set to 80, but your input_length is only 66. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=33)


Battery life is excellent and lasts almost two days. Camera quality is impressive in daylight. Charging speed is quite fast. Gaming performance could have been better.


In [10]:
def create_chunks(texts, chunk_size=3):
    chunks = []

    for i in range(0, len(texts), chunk_size):
        chunks.append(texts[i:i + chunk_size])

    return chunks


review_chunks = create_chunks(customer_reviews, chunk_size=3)

print("Total chunks:", len(review_chunks))

for i, chunk in enumerate(review_chunks):
    print(f"\nChunk {i + 1}:")
    print(chunk)

Total chunks: 3

Chunk 1:
['Battery life is excellent and lasts almost two days.', 'The display is bright and has beautiful colors.', 'Camera quality is impressive in daylight.']

Chunk 2:
['The phone heats up during heavy gaming.', 'Gaming performance could have been better.', 'The phone has a premium design.']

Chunk 3:
['Charging speed is quite fast.', 'Overall, a good phone for everyday usage.']


In [11]:
chunk_summaries = []

for i, chunk in enumerate(review_chunks):

    chunk_text = " ".join(chunk)

    result = summarizer(
        chunk_text,
        max_length=60,
        min_length=15,
        do_sample=False,
        truncation=True
    )

    chunk_summaries.append(result[0]["summary_text"])

    print(f"Chunk {i + 1} Summary:")
    print(result[0]["summary_text"])

Your max_length is set to 60, but your input_length is only 28. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=14)
Your max_length is set to 60, but your input_length is only 24. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=12)


Chunk 1 Summary:
Display is bright and has beautiful colors. Camera quality is impressive in daylight. Battery life is excellent.


Your max_length is set to 60, but your input_length is only 18. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=9)


Chunk 2 Summary:
The phone heats up during heavy gaming. Gaming performance could have been better. The phone has a premium design.
Chunk 3 Summary:
Review: Overall, a good phone for everyday usage. Charging speed is quite fast.


In [12]:
combined_summary = " ".join(chunk_summaries)

final_summary = summarizer(
    combined_summary,
    max_length=100,
    min_length=25,
    do_sample=False,
    truncation=True
)

print("FINAL PRODUCT SUMMARY:")
print(final_summary[0]["summary_text"])

Your max_length is set to 100, but your input_length is only 62. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=31)


FINAL PRODUCT SUMMARY:
Display is bright and has beautiful colors. Camera quality is impressive in daylight. Charging speed is quite fast. Gaming performance could have been better.


In [14]:
from rouge_score import rouge_scorer

reference_summary = """
The smartphone has excellent battery life, a vibrant display,
and good camera quality. However, gaming performance and
heating remain concerns.
"""

generated_summary = final_summary[0]["summary_text"]

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

scores = scorer.score(
    reference_summary,
    generated_summary
)

for metric, score in scores.items():
    print(f"\n{metric.upper()}")
    print("Precision:", round(score.precision, 4))
    print("Recall:", round(score.recall, 4))
    print("F1 Score:", round(score.fmeasure, 4))


ROUGE1
Precision: 0.2917
Recall: 0.35
F1 Score: 0.3182

ROUGE2
Precision: 0.087
Recall: 0.1053
F1 Score: 0.0952

ROUGEL
Precision: 0.25
Recall: 0.3
F1 Score: 0.2727


In [19]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("Gemini API key not found!")

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


In [22]:
MODEL_ID = "gemini-3.5-flash-lite"

In [23]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Explain customer sentiment analysis in simple terms."
)

print(response.text)

Imagine you run a bakery. Every day, people leave reviews, post on social media, or tell you what they think of your cupcakes. 

Instead of reading every single comment yourself to figure out if people are generally happy, angry, or just "meh," you hire a very fast, very smart assistant. That assistant reads everything in seconds and gives you a summary: *“Today, 80% of customers love the chocolate frosting, but 20% are mad that the cinnamon rolls are too dry.”*

**Customer sentiment analysis** is basically that smart assistant, but it’s done by computers and AI.

Here is how it works in 3 simple steps:

1. **Gather the Feedback:** The computer collects what customers are saying—tweets, reviews, emails, or chat logs.
2. **Analyze the Emotion:** The computer reads the text and looks at "feeling words." 
   * *“The service was **amazing**!”* = Positive
   * *“This is the **worst** product ever.”* = Negative
   * *“It arrived on Tuesday.”* = Neutral
3. **Report the Results:** It puts all 

In [24]:
SYSTEM_PROMPT = """
You are RetailMind, an intelligent product review assistant.

Your responsibilities:
1. Analyze customer product reviews.
2. Identify positive and negative feedback.
3. Identify recurring customer complaints.
4. Answer questions using the provided review data.
5. Provide concise and easy-to-understand responses.

Rules:
- Only use the information provided in the product reviews.
- Never invent product specifications or customer opinions.
- If the information is unavailable, clearly say so.
- Maintain a professional and helpful tone.
"""

In [25]:
product_reviews = """
Product: Smartphone X

Reviews:
1. Excellent battery life.
2. Camera quality is impressive.
3. The phone heats up during gaming.
4. Display quality is very good.
5. Battery lasts almost two days.
6. Gaming performance is disappointing.
7. Premium design and comfortable grip.
"""

In [26]:
user_question = "What are the biggest strengths and weaknesses of this smartphone?"

response = client.models.generate_content(
    model=MODEL_ID,
    contents=f"""
    {SYSTEM_PROMPT}

    PRODUCT REVIEWS:
    {product_reviews}

    USER QUESTION:
    {user_question}
    """
)

print(response.text)

Based on the provided product reviews, here are the strengths and weaknesses of Smartphone X:

**Strengths:**
* **Battery Life:** Excellent battery life that lasts almost two days.
* **Camera:** Impressive camera quality.
* **Display:** Very good display quality.
* **Design:** Premium design with a comfortable grip.

**Weaknesses:**
* **Temperature:** The phone heats up during gaming.
* **Gaming Performance:** Disappointing gaming performance.


In [27]:
def retailmind_chatbot(question, product_reviews):

    prompt = f"""
    {SYSTEM_PROMPT}

    PRODUCT REVIEWS:
    {product_reviews}

    USER QUESTION:
    {question}
    """

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt
    )

    return response.text

In [28]:
question = "What do customers dislike about this smartphone?"

answer = retailmind_chatbot(
    question,
    product_reviews
)

print(answer)

Based on the provided product reviews, customers dislike the following:

* The phone heats up during gaming.
* Gaming performance is disappointing. 

If you have any more questions about the reviews, feel free to ask!


In [29]:
questions = [
    "What are the biggest strengths of this smartphone?",
    "What are the common complaints?",
    "Is the battery life good?",
    "What do customers say about gaming performance?"
]

for question in questions:

    print("\nQuestion:", question)

    answer = retailmind_chatbot(
        question,
        product_reviews
    )

    print("Answer:", answer)


Question: What are the biggest strengths of this smartphone?
Answer: Hello! As your RetailMind assistant, I would be happy to help answer your question based on the provided product reviews.

Based on the reviews, the biggest strengths of Smartphone X are:
* **Battery life:** Customers noted it is excellent and lasts almost two days.
* **Camera:** The camera quality is impressive.
* **Display:** The display quality is very good.
* **Design:** It features a premium design and a comfortable grip.

Please let me know if you have any other questions about Smartphone X!

Question: What are the common complaints?
Answer: Based on the provided product reviews, the common complaints about Smartphone X are:

* The phone heats up during gaming.
* Gaming performance is disappointing.

Question: Is the battery life good?
Answer: Hello! I am RetailMind, your intelligent product review assistant.

Based on the provided product reviews, yes, the battery life is good. Specifically, the reviews note t

In [30]:
PROMPT_TEMPLATE = """
You are RetailMind, an intelligent retail product support assistant.

PRODUCT INFORMATION:
Product Name: {product_name}
Product Metadata: {product_metadata}

CUSTOMER REVIEWS:
{product_reviews}

USER QUESTION:
{question}

INSTRUCTIONS:
1. Answer using the supplied product information and reviews.
2. Do not invent product specifications.
3. Distinguish product facts from customer opinions.
4. If information is unavailable, clearly state that.
5. Keep answers concise, helpful and professional.

ANSWER:
"""

In [31]:
product_name = "Smartphone X"

product_metadata = {
    "brand": "ExampleTech",
    "category": "Smartphone",
    "battery": "5000 mAh",
    "display": "AMOLED",
    "price": "₹25,000"
}

In [32]:
def build_prompt(
    product_name,
    product_metadata,
    product_reviews,
    question
):
    return PROMPT_TEMPLATE.format(
        product_name=product_name,
        product_metadata=product_metadata,
        product_reviews=product_reviews,
        question=question
    )

In [33]:
prompt = build_prompt(
    product_name=product_name,
    product_metadata=product_metadata,
    product_reviews=product_reviews,
    question="Does this smartphone have good battery life?"
)

print(prompt)


You are RetailMind, an intelligent retail product support assistant.

PRODUCT INFORMATION:
Product Name: Smartphone X
Product Metadata: {'brand': 'ExampleTech', 'category': 'Smartphone', 'battery': '5000 mAh', 'display': 'AMOLED', 'price': '₹25,000'}

CUSTOMER REVIEWS:

Product: Smartphone X

Reviews:
1. Excellent battery life.
2. Camera quality is impressive.
3. The phone heats up during gaming.
4. Display quality is very good.
5. Battery lasts almost two days.
6. Gaming performance is disappointing.
7. Premium design and comfortable grip.


USER QUESTION:
Does this smartphone have good battery life?

INSTRUCTIONS:
1. Answer using the supplied product information and reviews.
2. Do not invent product specifications.
3. Distinguish product facts from customer opinions.
4. If information is unavailable, clearly state that.
5. Keep answers concise, helpful and professional.

ANSWER:



In [34]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt
)

print(response.text)

Hello! I am RetailMind, your retail product support assistant. 

Based on the product specifications and customer reviews for the **Smartphone X** (Brand: ExampleTech, Price: ₹25,000):

* **Product Fact:** The Smartphone X is equipped with a **5000 mAh battery**.
* **Customer Opinion:** Customers consistently praise the battery life, noting that it is "excellent" and "lasts almost two days."

Yes, based on the specifications and customer feedback, this smartphone has a very good battery life.


In [35]:
from google.genai import errors

def retailmind_chatbot(
    question,
    product_name,
    product_metadata,
    product_reviews
):
    prompt = build_prompt(
        product_name,
        product_metadata,
        product_reviews,
        question
    )

    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt
        )

        return response.text

    except errors.APIError as e:
        print("Gemini API Error:", e)
        return "Unable to process your request right now."

In [36]:
answer = retailmind_chatbot(
    question="What are the main customer complaints?",
    product_name=product_name,
    product_metadata=product_metadata,
    product_reviews=product_reviews
)

print(answer)

Based on the provided customer reviews, the main customer complaints regarding Smartphone X are:

1. **Heating Issues:** The phone tends to heat up during gaming.
2. **Gaming Performance:** The gaming performance is considered disappointing. 

*(Note: These points reflect customer opinions and experiences shared in the reviews.)*


In [ ]:
import assistant
reviews = [
    "Battery life is excellent.",
    "Camera quality is impressive.",
    "The phone heats up during gaming.",
    "Display quality is amazing.",
    "Gaming performance is disappointing."
]

summary = assistant.summarize_reviews(reviews)

print("PRODUCT SUMMARY:")
print(summary)

Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Your max_length is set to 80, but your input_length is only 19. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=9)
Your max_length is set to 80, but your input_length is only 12. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=6)
Your max_length is set to 100, but your input_length is only 48. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=24)


PRODUCT SUMMARY:
Display quality is great, but performance is still disappointing. Camera quality is impressive. Battery life is excellent. The phone heats up during gaming.


In [45]:
import importlib
import assistant

importlib.reload(assistant)

<module 'assistant' from 'd:\\RetailMind\\assistant.py'>

In [47]:
client,chat = assistant.create_product_chatbot(
    product_name="Smartphone X",
    product_metadata={
        "battery": "5000 mAh",
        "display": "AMOLED",
        "price": "₹25,000"
    },
    product_reviews=reviews
)

response = chat.send_message(
    "What are the biggest customer complaints?"
)

print("CHATBOT RESPONSE:")
print(response.text)

CHATBOT RESPONSE:
Based on customer reviews, the primary complaints regarding the Smartphone X are:

* **Thermal issues:** Customers report that the phone heats up during gaming.
* **Gaming performance:** The gaming performance is described by users as disappointing. 

Please note that these points represent customer opinions and experiences rather than official product specifications.


In [48]:
sample_reviews = [
    "Battery life is excellent.",
    "Camera quality is impressive.",
    "The phone heats up during gaming.",
    "Display quality is amazing.",
    "Gaming performance is disappointing."
]

summary = assistant.summarize_reviews(sample_reviews)

assert isinstance(summary, str)
assert summary.strip() != ""

print("Summarization: PASSED")
print(summary)

Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Your max_length is set to 80, but your input_length is only 19. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=9)
Your max_length is set to 80, but your input_length is only 12. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=6)
Your max_length is set to 100, but your input_length is only 48. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=24)


Summarization: PASSED
Display quality is great, but performance is still disappointing. Camera quality is impressive. Battery life is excellent. The phone heats up during gaming.


In [49]:
client.close()

print("Gemini client closed.")

Gemini client closed.
